# SimpliPy Engine Memory Consumption
Load the `dev_7-3` SimpliPy engine and measure how much memory it uses.

In [1]:
import psutil
import os
import sys
import tracemalloc
from simplipy import SimpliPyEngine

## Baseline Memory Usage

In [2]:
process = psutil.Process(os.getpid())
baseline = process.memory_info()
print(f"Baseline RSS: {baseline.rss / 1024 / 1024:.2f} MB")
print(f"Baseline VMS: {baseline.vms / 1024 / 1024:.2f} MB")

tracemalloc.start()

Baseline RSS: 127.23 MB
Baseline VMS: 3355.43 MB


## Load the dev_7-3 SimpliPy Engine

In [3]:
engine = SimpliPyEngine.load('dev_7-3', install=True)
print(f"Engine loaded: {engine}")

Engine loaded: <simplipy.engine.SimpliPyEngine object at 0x7f32feab2120>


## Memory Consumption

In [4]:
post_load = process.memory_info()

rss_diff = (post_load.rss - baseline.rss) / 1024 / 1024
vms_diff = (post_load.vms - baseline.vms) / 1024 / 1024

print(f"Post-load RSS: {post_load.rss / 1024 / 1024:.2f} MB")
print(f"Post-load VMS: {post_load.vms / 1024 / 1024:.2f} MB")
print()
print(f"RSS increase:  {rss_diff:.2f} MB")
print(f"VMS increase:  {vms_diff:.2f} MB")
print()

current, peak = tracemalloc.get_traced_memory()
print(f"Traced current: {current / 1024 / 1024:.2f} MB")
print(f"Traced peak:    {peak / 1024 / 1024:.2f} MB")

Post-load RSS: 539.40 MB
Post-load VMS: 3841.56 MB

RSS increase:  412.18 MB
VMS increase:  486.13 MB

Traced current: 94.27 MB
Traced peak:    129.40 MB


## Detailed Memory Breakdown

In [5]:
snapshot = tracemalloc.take_snapshot()
top_stats = snapshot.statistics('lineno')

print("Top 15 memory allocations:")
for stat in top_stats[:15]:
    print(f"  {stat}")

print()
print(f"sys.getsizeof(engine): {sys.getsizeof(engine)} bytes")

tracemalloc.stop()

Top 15 memory allocations:
  /home/psaegert/miniconda3/envs/flash-ansr/lib/python3.13/json/decoder.py:361: size=26.4 MiB, count=582818, average=47 B
  /home/psaegert/miniconda3/envs/flash-ansr/lib/python3.13/site-packages/simplipy/utils.py:540: size=9592 KiB, count=113998, average=86 B
  /home/psaegert/miniconda3/envs/flash-ansr/lib/python3.13/site-packages/simplipy/engine.py:847: size=8873 KiB, count=283928, average=32 B
  /home/psaegert/miniconda3/envs/flash-ansr/lib/python3.13/site-packages/simplipy/engine.py:862: size=7547 KiB, count=214660, average=36 B
  /home/psaegert/miniconda3/envs/flash-ansr/lib/python3.13/site-packages/simplipy/utils.py:548: size=7125 KiB, count=114002, average=64 B
  /home/psaegert/miniconda3/envs/flash-ansr/lib/python3.13/site-packages/simplipy/engine.py:200: size=6234 KiB, count=114000, average=56 B
  /home/psaegert/miniconda3/envs/flash-ansr/lib/python3.13/site-packages/simplipy/engine.py:850: size=5870 KiB, count=107330, average=56 B
  /home/psaegert/mi

## Comparison: SymPy Import Memory
Since SymPy has no engine object, the fairest comparison is the memory cost of `import sympy` — which loads all its simplification rules, pattern matching, and core algebra into memory. We measure this in a subprocess to get an isolated reading.

In [6]:
import subprocess, json

# Measure SymPy import memory in an isolated subprocess
script = """
import psutil, os, json, tracemalloc

process = psutil.Process(os.getpid())
baseline = process.memory_info()
tracemalloc.start()

import sympy  # noqa: E402

post = process.memory_info()
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(json.dumps({
    "baseline_rss": baseline.rss,
    "baseline_vms": baseline.vms,
    "post_rss": post.rss,
    "post_vms": post.vms,
    "rss_diff": post.rss - baseline.rss,
    "vms_diff": post.vms - baseline.vms,
    "traced_current": current,
    "traced_peak": peak,
    "sympy_version": sympy.__version__,
}))
"""

result = subprocess.run(
    [sys.executable, "-c", script],
    capture_output=True, text=True
)
sympy_mem = json.loads(result.stdout)

print(f"SymPy version: {sympy_mem['sympy_version']}")
print(f"RSS increase:   {sympy_mem['rss_diff'] / 1024 / 1024:.2f} MB")
print(f"VMS increase:   {sympy_mem['vms_diff'] / 1024 / 1024:.2f} MB")
print(f"Traced current: {sympy_mem['traced_current'] / 1024 / 1024:.2f} MB")
print(f"Traced peak:    {sympy_mem['traced_peak'] / 1024 / 1024:.2f} MB")

SymPy version: 1.14.0
RSS increase:   58.10 MB
VMS increase:   58.05 MB
Traced current: 31.53 MB
Traced peak:    31.56 MB


## Side-by-Side Comparison

In [7]:
simplipy_rss = rss_diff
sympy_rss = sympy_mem['rss_diff'] / 1024 / 1024

simplipy_traced = peak / 1024 / 1024
sympy_traced = sympy_mem['traced_peak'] / 1024 / 1024

print(f"{'Metric':<25} {'SimpliPy dev_7-3':>18} {'SymPy':>18} {'Ratio':>10}")
print("-" * 75)
print(f"{'RSS increase (MB)':<25} {simplipy_rss:>18.2f} {sympy_rss:>18.2f} {simplipy_rss / sympy_rss:>9.1f}x")
print(f"{'Traced peak (MB)':<25} {simplipy_traced:>18.2f} {sympy_traced:>18.2f} {simplipy_traced / sympy_traced:>9.1f}x")

Metric                      SimpliPy dev_7-3              SymPy      Ratio
---------------------------------------------------------------------------
RSS increase (MB)                     412.18              58.10       7.1x
Traced peak (MB)                      129.40              31.56       4.1x
